# Baseline Model Convolutional Neural Network

### Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Part 1 - Data Preprocessing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf

img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/faces_split/train",
    image_size=img_size,
    batch_size=batch_size,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/faces_split/val",
    image_size=img_size,
    batch_size=batch_size,
    label_mode="binary"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/faces_split/test",
    image_size=img_size,
    batch_size=batch_size,
    label_mode="binary",
    shuffle=False
)

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

## Part 2 - Building the CNN

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

## Part 3 - Training the CNN

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(train_ds, validation_data=val_ds, epochs=10)

## Part 4 - Evaluating the Model

In [ ]:
model.evaluate(test_ds)

## Part 5 - Making a single prediction

In [ ]:
import cv2
import numpy as np

def predict_video_faces(video_path, model, img_size=(224, 224), frame_skip=10):

    face_detector = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )

    cap = cv2.VideoCapture(video_path)

    frames = []
    frame_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id % frame_skip == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            faces = face_detector.detectMultiScale(gray, 1.1, 5)

            if len(faces) > 0:
                # largest face
                faces = sorted(faces, key=lambda x: x[2]*x[3], reverse=True)
                x, y, w, h = faces[0]

                face = frame[y:y+h, x:x+w]
                face = cv2.resize(face, img_size)
                face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
                face = face.astype("float32") / 255.0

                frames.append(face)

        frame_id += 1

    cap.release()

    if len(frames) == 0:
        print("No faces detected in video")
        return

    frames = np.array(frames)

    # predict
    preds = model.predict(frames, verbose=0).flatten()

    # average score
    video_score = preds.mean()

    # final label
    label = "fake" if video_score >= 0.5 else "real"

    # confidence
    confidence = video_score if label == "fake" else 1 - video_score

    return {
    "label": label,
    "score": float(video_score),
    "confidence": float(confidence),
    "frames_used": len(frames)
          }

In [ ]:
prediction = predict_video_faces("/content/new_video.mp4", model)
print(prediction)